# Typed domains and legal moves

The polar Fourier transform is a chain of three steps: an angular DFT, one discrete Hankel transform (DHT) per harmonic, and an angular inverse DFT. Between the steps, a polar array sits at one of four points, its **domains**. This notebook names those domains, wraps an array together with the domain it is in, and walks the chain one step at a time, plotting the signal after every step.

```text
POLAR_SPATIAL --DFT--> POLAR_SPATIAL_HARMONIC --DHT--> POLAR_FREQUENCY_HARMONIC --IDFT--> POLAR_FREQUENCY
```

Every `pypft.Domain` member starts with `POLAR`, since every point on the chain is sampled on the same `pypft.PolarGrid`. The word after it is the radial coordinate: `SPATIAL` for the radius $r$, `FREQUENCY` for the radial frequency $\rho$; only the DHT changes it. A trailing `HARMONIC` marks the angular coordinate as a harmonic order $n$ rather than a physical angle; only the angular DFT and its inverse change it.

In [ ]:
# NumPy for arrays, matplotlib for figures, and PyPFT itself
import matplotlib.pyplot as plt
import numpy as np

import pypft

## Wrapping a signal

`pypft.BaseSignal` has one subclass per domain. Each wraps a `(values, grid)` pair, where `values` has PyPFT's `(radial, angular)` layout, and records which domain the values are in. A signal sampled in space starts as a `pypft.PolarSpatialSignal`.

The test signal is a Gaussian of width $\sigma$ centered off the origin, at radius $r_0$ and angle $\theta_0$. In polar coordinates the squared distance to that center is $r^2 + r_0^2 - 2 r r_0 \cos(\theta - \theta_0)$, so

$$f(r, \theta) = \exp\!\left(-\frac{r^2 + r_0^2 - 2 r r_0 \cos(\theta - \theta_0)}{2\sigma^2}\right).$$

Being off center, it depends on the angle, so every harmonic carries something. `grid.r.T` has one column of radii per angular sample and `grid.theta` one angle per column, matching the `(radial, angular)` layout.

In [ ]:
# A grid of 64 radial and 31 angular samples within the space limit R = 8
grid = pypft.PolarGrid(n_radial=64, n_angular=31, R=8.0)

# Radii and angles on the (radial, angular) layout
r = grid.r.T
theta = grid.theta[np.newaxis, :]

# A Gaussian of width 1, centered at radius 3 and angle pi/4
r0, theta0, sigma = 3.0, np.pi / 4, 1.0
squared_distance = r**2 + r0**2 - 2 * r * r0 * np.cos(theta - theta0)
f = np.exp(-squared_distance / (2 * sigma**2))

# Wrap the samples with the domain they are in
signal = pypft.PolarSpatialSignal(values=f, grid=grid)
print(signal.domain)

## Walking one step at a time

Each subclass has a step method for each neighbor of its domain on the chain, named after the destination: `to_polar_spatial_harmonic`, `to_polar_frequency_harmonic`, `to_polar_frequency`, and, going back, `to_polar_spatial`. Chaining them reads as the chain itself.

The cell walks the three forward steps and plots every signal on the way with `signal.plot()`, which draws the magnitude and the phase of the values on the polar index table (radial index up, angular index across):

- `POLAR_SPATIAL`: the off-center Gaussian, a bright band across the angular columns near $\theta_0$.
- `POLAR_SPATIAL_HARMONIC`: the angular DFT of every radius. The columns are now harmonics; a function concentrated in angle spreads over many harmonics.
- `POLAR_FREQUENCY_HARMONIC`: each harmonic's Hankel transform, now indexed by radial frequency.
- `POLAR_FREQUENCY`: back to a physical angle, now the angle of the frequency vector.

In [ ]:
# Walk the chain one step at a time, keeping every signal on the way
harmonic_signal = signal.to_polar_spatial_harmonic()
frequency_harmonic_signal = harmonic_signal.to_polar_frequency_harmonic()
frequency_signal = frequency_harmonic_signal.to_polar_frequency()
chain = (signal, harmonic_signal, frequency_harmonic_signal, frequency_signal)

# One row per domain: magnitude on the left, phase on the right
fig, axes = plt.subplots(nrows=len(chain), ncols=2, figsize=(10, 13))
for step, row in zip(chain, axes):
    step.plot(ax=(row[0], row[1]))
fig.tight_layout()
plt.show()

Each step returns a signal of a different, statically known type, so the type of the signal tells where it is on the chain:

In [ ]:
# Every step returns the subclass of its destination domain
for step in chain:
    print(f"{type(step).__name__:>30}: {step.domain}")

## Legal moves only

A subclass defines step methods only for the neighbors of its domain. `PolarSpatialHarmonicSignal` sits in the middle of the chain, so it has `to_polar_spatial` and `to_polar_frequency_harmonic`, and no `to_polar_frequency`: that move would skip the DHT. A type checker such as `pyright` rejects the call before the code runs, and at run time it raises `AttributeError`:

In [ ]:
# Skipping a step is not a method of this subclass
try:
    harmonic_signal.to_polar_frequency()  # type: ignore[attr-defined]
except AttributeError as exc:
    print(exc)

Stepping back undoes a step. Going back from `POLAR_SPATIAL_HARMONIC` to `POLAR_SPATIAL` is the inverse angular DFT, which returns the original samples:

In [ ]:
# One step forward and one step back
back = harmonic_signal.to_polar_spatial()

# The original samples, up to rounding
print(f"domain: {back.domain}")
print(f"max |back - f|: {np.abs(back.values - f).max():.1e}")

## Walking to any domain

When the target domain is only known at run time, `to(domain)` walks the chain toward it, one legal step at a time, in whichever direction it lies. Walking from `POLAR_SPATIAL` to `POLAR_FREQUENCY` gives the same values as the three explicit steps:

In [ ]:
# Walk to the far end of the chain in one call
walked = signal.to(domain=pypft.Domain.POLAR_FREQUENCY)

# The same values as the explicit walk
print(f"domain: {walked.domain}")
print(f"max difference: {np.abs(walked.values - frequency_signal.values).max():.1e}")

A target that is not a `Domain` member is rejected straight away:

In [ ]:
# A string is not a Domain member
try:
    signal.to(domain="POLAR_FREQUENCY")  # type: ignore[arg-type]
except TypeError as exc:
    print(exc)

## Where to go next

Walking from `POLAR_SPATIAL` to `POLAR_FREQUENCY` is the whole forward polar Fourier transform. [05_pft_and_ipft](05_pft_and_ipft.ipynb) introduces `pypft.forward_pft` and `pypft.inverse_pft`, the one-call shortcuts that take and return plain arrays, and checks them against an exact transform pair.